# Figure 1D sparse-cache import

Run the next cell before inspecting the matrix or metadata. It reads the validated per-cell metadata cache and aligns it to the matrix barcode order.

In [ ]:
library(Matrix)
library(reticulate)

# This notebook is stored in New_Figure_1D, one level below the project root.
sparse_dir <- file.path("..", "results", "figure1d", "sparse_cache")
genes <- readLines(file.path(sparse_dir, "GSE200164_genes.txt"))
barcodes <- readLines(file.path(sparse_dir, "GSE200164_barcodes.txt"))

# Import and explicitly align the one-row-per-cell metadata table.
metadata <- utils::read.csv(
  file.path(sparse_dir, "GSE200164_cell_metadata.csv"),
  stringsAsFactors = FALSE,
  check.names = FALSE
)
stopifnot("cell" %in% colnames(metadata), !anyDuplicated(metadata$cell))
rownames(metadata) <- metadata$cell
metadata <- metadata[barcodes, , drop = FALSE]
stopifnot(identical(rownames(metadata), barcodes), !anyNA(metadata$sample_id))

# Load the SciPy CSR cache without densifying it, then convert to Seurat-friendly CSC.
Sys.setenv(RETICULATE_PYTHON = Sys.which("python3"))
scipy_sparse <- import("scipy.sparse", convert = FALSE)
counts_python <- scipy_sparse$load_npz(file.path(sparse_dir, "GSE200164_counts_csr.npz"))
counts <- py_to_r(counts_python)
rm(counts_python)
counts <- as(counts, "CsparseMatrix")
rownames(counts) <- genes
colnames(counts) <- barcodes

cat("Matrix class after conversion:", class(counts)[1], "\n")
cat("First 6 genes:\n")
print(head(rownames(counts)))
cat("\nFirst 6 matrix barcodes:\n")
print(head(colnames(counts)))
cat("\nMetadata column names:\n")
print(colnames(metadata))
